In [18]:
import pandas as pd
import bibtexparser
import glob
import os
import arxiv
import time

def get_arxiv_info(arxiv_id):
    """
    Fetch paper information from arXiv using the arxiv ID.
    Includes rate limiting to be respectful to arXiv's servers.
    
    Args:
        arxiv_id: arXiv identifier (can be with or without version number)
    
    Returns:
        dict: Dictionary with arxiv information or None if not found
    """
    print('arxiv_id: ', arxiv_id)
    try:
        # Remove version number if present (e.g., v1, v2)
        clean_id = arxiv_id.split('v')[0]
        
        # Add rate limiting
        time.sleep(3)  # Be nice to arXiv servers
        
        # Create client and search for the paper
        client = arxiv.Client()
        search = arxiv.Search(id_list=[clean_id])
        paper = next(client.results(search))
        
        # Extract version from paper URL if present
        version = paper.entry_id.split('v')[-1] if 'v' in paper.entry_id else '1'
        
        return {
            'arxiv_url': paper.entry_id,
            'arxiv_primary_category': paper.primary_category,
            'arxiv_version': version,
            'abstract': paper.summary,
            'published_date': paper.published.strftime('%Y-%m-%d') if paper.published else None
        }
    except Exception as e:
        print(f"Error fetching arXiv info for {arxiv_id}: {str(e)}")
        return None

def extract_bib_info(entry, collaboration):
    """
    Extract relevant information from a BibTeX entry.
    
    Args:
        entry: BibTeX entry dictionary
        collaboration: Name of collaboration extracted from filename
    
    Returns:
        dict: Dictionary with extracted information
    """
    # Initialize with default values
    info = {
        'collaboration': collaboration,
        'title': '',
        'bibtag': '',
        'eprint': '',
        'year': '',
        'interaction': None,
        'target': None,
        'arxiv_url': None,
        'arxiv_primary_category': None,
        'arxiv_version': None,
        'abstract': None,
        'authors': None,
        'published_date': None
    }
    
    # Extract bibtag (ID/key of the entry)
    info['bibtag'] = entry.get('ID', '')
    
    # Extract title (remove curly braces if present)
    title = entry.get('title', '')
    info['title'] = title.replace('{', '').replace('}', '')
    
    # Extract year and ensure it's numeric
    year = entry.get('year', '')
    try:
        info['year'] = int(year) if year else None
    except ValueError:
        info['year'] = None
    
    # Extract eprint (arxiv number)
    info['eprint'] = entry.get('eprint', '')
    
    # # If we have an eprint, fetch arXiv information
    # if info['eprint']:
    #     arxiv_info = get_arxiv_info(info['eprint'])
    #     if arxiv_info:
    #         info.update(arxiv_info)
    
    return info

def process_bib_files(directory_path='*.bib'):
    """
    Process all .bib files in the specified directory and create a DataFrame.
    Fetches additional information from arXiv when available.
    
    Args:
        directory_path: Path pattern for .bib files (default: '*.bib' in current directory)
    
    Returns:
        pandas.DataFrame: DataFrame containing information from all .bib files and arXiv
    """
    all_entries = []
    
    # Find all .bib files
    bib_files = glob.glob(directory_path)

    cnt=0
    for bib_file in bib_files:
        # Extract collaboration name from filename (remove .bib extension)
        collaboration = os.path.splitext(os.path.basename(bib_file))[0]
        
        try:
            # Read the .bib file
            with open(bib_file, 'r', encoding='utf-8') as bibtex_file:
                parser = bibtexparser.bparser.BibTexParser()
                bib_database = bibtexparser.load(bibtex_file, parser=parser)
            
            # Process each entry in the file
            for entry in bib_database.entries:
                entry_info = extract_bib_info(entry, collaboration)
                all_entries.append(entry_info)
                
        except Exception as e:
            print(f"Error processing {bib_file}: {str(e)}")
    
    # Create DataFrame
    df = pd.DataFrame(all_entries)
    
    # Ensure all required columns are present
    required_columns = ['collaboration', 'title', 'bibtag', 'eprint', 'year',
                       'interaction', 'target', 'arxiv_url', 'arxiv_primary_category',
                       'arxiv_version', 'abstract', 'authors', 'published_date']
    
    for col in required_columns:
        if col not in df.columns:
            df[col] = None
    
    # Sort by collaboration and year (descending) and reset index
    df = df[required_columns].sort_values(['collaboration', 'year'], 
                                        ascending=[True, False],
                                        na_position='last').reset_index(drop=True)
    
    return df

In [19]:
df = process_bib_files('*.bib')

df.to_csv('bibliography_database.csv', index=False)

In [20]:
df = pd.read_csv('bibliography_database.csv')

In [21]:
df

,collaboration,title,bibtag,eprint,year,interaction,target,arxiv_url,arxiv_primary_category,arxiv_version,abstract,authors,published_date
0,ArgoNeuT,First measurement of electron neutrino scatter...,ArgoNeuT:2020kir,2004.01956,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ArgoNeuT,First measurement of the cross section for $\n...,ArgoNeuT:2018und,1804.10294,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ArgoNeuT,Measurement of $\nu_\mu$ and $\bar\nu_\mu$ neu...,ArgoNeuT:2015ldo,1511.00941,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ArgoNeuT,Detection of Back-to-Back Proton Pairs in Char...,ArgoNeuT:2014ihi,1405.4261,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ArgoNeuT,First Measurement of Neutrino and Antineutrino...,ArgoNeuT:2014uwh,1408.0598,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,T2K,Measurement of the electron neutrino charged-c...,T2K:2015ydf,1503.08815,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138,T2K,Measurement of the Inclusive Electron Neutrino...,T2K:2014lbi,1407.7389,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
139,T2K,Measurement of the inclusive $\nu_\mu$ charged...,T2K:2014axs,1407.4256,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
140,T2K,Measurement of the neutrino-oxygen neutral-cur...,T2K:2014vog,1403.3140,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
import pandas as pd

def classify_references():
    # Dictionary to store reference classifications
    classifications = {}
    
    # Table 1 - Inclusive measurements with neutrino flavors
    inclusive_refs = {
        'ArgoNeuT:2020kir': 'νe, ν̄e',
        'ArgoNeuT:2011bms': 'νμ',
        'ArgoNeuT:2014rlj': 'νμ, ν̄μ',
        'MicroBooNE:2024zkh':  'νμ',
        'MicroBooNE:2024zwf':  'νμ',
        'MicroBooNE:2023foc':  'νμ',
        'MicroBooNE:2019nio': 'νμ',
        'MicroBooNE:2021sfa': 'νμ',
        'MicroBooNE:2021ppm': 'νe, ν̄e',
        'MicroBooNE:2021gfj': 'νe, ν̄e',
        'MiniBooNE:2018dus': 'νμ monoenergetic K+ decay',
        'MINERvA:2020zzv': 'νμ',
        'MINERvA:2021owq': 'νμ',
        'MINERvA:2021wjs': 'νμ',
        'MINERvA:2023ner': 'νe, ν̄e',
        'MINERvA:2014rdw': 'νμ',
        'MINERvA:2015ydy': 'νμ',
        'MINERvA:2016oql': 'νμ',
        'MINERvA:2016ing': 'νμ, ν̄μ',
        'MINERvA:2017ozn': 'νμ/ν̄μ',
        'MINERvA:2019wqe': 'ν̄μ',
        'MINOS:2009ugl': 'νμ, ν̄μ',
        'NOMAD:2007krq': 'νμ',
        'SciBooNE:2010slc': 'νμ',
        'NOvA:2024rov': 'νμ',
        'NOvA:2024zmr': 'νμ',
        'NOvA:2021eqi': 'νμ',
        'NOvA:2022see': 'νe',
        'T2K:2020lrr': 'ν̄e',
        'T2K:2014lbi': 'νe',
        'T2K:2015ydf': 'νe',
        'T2K:2017bvo': 'νμ/ν̄μ',
        'T2K:2013nor': 'νμ',
        'T2K:2018lnf': 'νμ',
        'T2K:2019dgm': 'νμ',
        'T2K:2014axs': 'νμ',
        'T2K:2015cxp': 'νμ',
        'NINJA:2020gbg': 'νμ + ν̄μ H2O',
        'NINJA:2020bvx': 'νμ Fe',
        'FASER:2024hoe': 'νe, νμ',   
    }
    
    # Table 2 - Quasielastic (without pions) with CC/NC information
    qe_refs = {
        'ArgoNeuT:2014ihi': 'CC (2p2h)',
        'K2K:2006odf': 'CC (MA)',
        'MicroBooNE:2020fxd': 'CC (differential)',
        'MINERvA:2019gsf': 'CC',
        'MINERvA:2019ope': 'CC',
        'MINERvA:2022mnw': 'CC',
        'MINERvA:2023ikp': 'CC (2n2h)',
        'MINERvA:2022bno': 'CC', 
        'MINERvA:2023kuz': 'CC',
        'MINERvA:2013kdn': 'CC (Q2)',
        'MINERvA:2013bcy': 'CC (Q2)',
        'MINERvA:2017dzh': 'CC (Q2)',
        'MINERvA:2014ypj': 'CC (1p)',
        'MINERvA:2015jih': 'CC (νe)',
        'MINERvA:2018hqn': 'CC (differential)',
        'MINERvA:2018vjb': 'CC (differential)',
        'MINERvA:2018hba': 'CC (differential)',
        'MicroBooNE:2020akw': 'CC 0π',
        'MicroBooNE:2023tzj': 'CCQE',
        'MicroBooNE:2023cmw': 'CCQE',
        'MicroBooNE:2023krv': 'CC 1mu1p',
        'MicroBooNE:2022emb': 'CC 2p2h',
        'MicroBooNE:2022tdd': 'CC νe', 
        'MicroBooNE:2024tmp': 'CC',
        'MiniBooNE:2010bsu': 'CC (differential)',
        'MiniBooNE:2013qnd': 'CC (differential)',
        'MiniBooNE:2007iti': 'CC (MA)',
        'MiniBooNE:2010xqw': 'NC',
        'MiniBooNE:2013dds': 'NC',
        'MINOS:2014axb': 'CC (MA)',
        'NOMAD:2009qmu': 'CC (MA, σ(Eν))',
        'Super-Kamiokande:2019hga': 'NC',
        'T2K:2020txr': 'CC ν̄μ, ν̄μ+νμ PM+WG+INGRID',
        'T2K:2023qjb': 'CC multi ND',
        'T2K:2016jor': 'CC (differential)',
        'T2K:2017qxv': 'CC (differential)',
        'T2K:2019ddy': 'CC (differential)',
        'T2K:2020sbd': 'CC (differential)',
        'T2K:2020jav': 'CC (differential)',
        'T2K:2015ujp': 'CC (σ(Eν))',
        'T2K:2014hih': 'CC (MA)',
        'T2K:2014vog': 'NC',
        'T2K:2019zqh': 'NC',
        'T2K:2018rnz': 'CC (differential)',
        'NINJA:2022zbi': 'CC event rates on Fe',
    }
    
    # Table 3 - Pion production with CC/NC information
    charged_pion_refs = {
        'MINERvA:2022esg': 'CC',
        'ArgoNeuT:2018und': 'CC',
        'ArgoNeuT:2014uwh': 'CC',
        'K2K:2008tus': 'CC',
        'K2K:2005uiu': 'CC',
        'MINERvA:2022djk' : 'CC',
        'MINERvA:2014ogb': 'CC',
        'MINERvA:2016sfc': 'CC',
        'MINERvA:2014ani': 'CC',
        'MINERvA:2019rhx': 'CC',
        'MINERvA:2017ipy': 'CC',
        'MiniBooNE:2010eis': 'CC',
        'MiniBooNE:2009koj': 'CC',
        'SciBooNE:2008bzb': 'CC',
        'T2K:2023xlh': 'CC Coh νμ+ν̄μ',
        'T2K:2021naz': 'CC TKI',
        'T2K:2016cbz': 'CC',
        'T2K:2016soz': 'CC',
        'T2K:2019yqu': 'CC',
        'NINJA:2022zbi': 'CC event rates on Fe',
        'NOMAD:2001vdk': 'CC bwd π- and p'
    }
    
    neutral_pion_refs = {
        'MicroBooNE:2024sec': 'NC',
        'MicroBooNE:2024bnl': 'CC',
        'ArgoNeuT:2015ldo': 'NC',
        'K2K:2010xeb': 'CC',
        'K2K:2004qpv': 'NC',
        'MicroBooNE:2018neo': 'CC',
        'MicroBooNE:2022zhr': 'NC',
        'MINERvA:2020anu': 'CC',
        'MINERvA:2016sfc': 'CC',
        'MINERvA:2015slz': 'CC',
        'MINERvA:2017okh': 'CC',
        'MINERvA:2016uck': 'NC',
        'MiniBooNE:2010cxl': 'CC',
        'MiniBooNE:2009dxl': 'NC',
        'MiniBooNE:2008mmr': 'NC',
        'MINOS:2016yyz': 'NC',
        'NOMAD:2009idt': 'NC',
        'NOvA:2019bdw': 'NC',
        'NOvA:2023uxq': 'NC',
        'SciBooNE:2009nlf': 'NC',
        'SciBooNE:2010lca': 'NC',
        'T2K:2017epu': 'NC'
    }

    exotic = {
        'MINERvA:2016ymg': 'NC K+',
        'MINERvA:2016zyp': 'CC K+',
        'MINERvA:2016cun': 'CC-Coh K+',
        'SciBooNE:2011sjq': 'CC K+',
        'T2K:2019odo': 'CC gamma',
        'MicroBooNE:2023ubu': 'CC+NC eta',
        'MicroBooNE:2022cls': 'CC Lambda',
        'NOMAD:2011gyy': 'CC gamma',
        'NOMAD:2004djf': 'NC K0, K+-, Λ, Λ ̄, Σ+-',
        'NOMAD:2001kdr': 'CC K0, K+-, Λ, Λ ̄, Σ+-, Σ0, Ξ-',
        'NOMAD:2001kpt': 'CC D+',
        'NOMAD:2001ozt': 'CC ρ0, f0, f2',
        'NOMAD:2001iup': 'CC Λ ̄',
        'NOMAD:2000wdf': 'CC Λ',
        'NOMAD:1998gsx': 'CC DIS',
    }
    
    # Create detailed classifications
    df_data = []
    
    # Process inclusive measurements
    for ref, flavor in inclusive_refs.items():
        df_data.append({
            'reference': ref,
            'topology': 'Inclusive',
            'details': f'ν flavor: {flavor}',
        })
    
    # Process quasielastic measurements
    for ref, interaction in qe_refs.items():
        df_data.append({
            'reference': ref,
            'topology': 'Without pions',
            'details': interaction,
        })
    
    # Process charged pion measurements
    for ref, interaction in charged_pion_refs.items():
        df_data.append({
            'reference': ref,
            'topology': 'With charged pion',
            'details': interaction,
        })
    
    # Process neutral pion measurements
    for ref, interaction in neutral_pion_refs.items():
        df_data.append({
            'reference': ref,
            'topology': 'With neutral pion',
            'details': interaction,
        })

    for ref, interaction in exotic.items():
        df_data.append({
            'reference': ref,
            'topology': 'Exotic',
            'details': interaction,
        })

    
    
    
    # Create DataFrame and sort
    df_new = pd.DataFrame(df_data)
    return df_new.sort_values(['reference', 'topology']).reset_index(drop=True)

# Create the DataFrame
df_new = classify_references()

In [6]:
df_new

,reference,topology,details
0,ArgoNeuT:2011bms,Inclusive,ν flavor: νμ
1,ArgoNeuT:2014ihi,Without pions,CC (2p2h)
2,ArgoNeuT:2014rlj,Inclusive,"ν flavor: νμ, ν̄μ"
3,ArgoNeuT:2014uwh,With charged pion,CC
4,ArgoNeuT:2015ldo,With neutral pion,NC
...,...,...,...
139,T2K:2020sbd,Without pions,CC (differential)
140,T2K:2020txr,Without pions,"CC ν̄μ, ν̄μ+νμ PM+WG+INGRID"
141,T2K:2021naz,With charged pion,CC TKI
142,T2K:2023qjb,Without pions,CC multi ND


In [7]:
df.bibtag

0      ArgoNeuT:2020kir
1      ArgoNeuT:2018und
2      ArgoNeuT:2015ldo
3      ArgoNeuT:2014ihi
4      ArgoNeuT:2014uwh
             ...       
137         T2K:2015ydf
138         T2K:2014lbi
139         T2K:2014axs
140         T2K:2014vog
141         T2K:2013nor
Name: bibtag, Length: 142, dtype: object

In [8]:
for x in df_new.reference:
    if x not in list(df.bibtag):
        print(x)
        #print(x, x in list(df_new.reference))

In [9]:
for x in df.bibtag:
    if x not in list(df_new.reference):
        print(x, x in list(df_new.reference))

NameError: name 'charged_pion_refs' is not defined

In [15]:
    # Table 3 - Pion production with CC/NC information
    charged_pion_refs = {
        'ArgoNeuT:2018und': 'CC',
        'ArgoNeuT:2014uwh': 'CC',
        'K2K:2008tus': 'CC',
        'K2K:2005uiu': 'CC',
        'MINERvA:2022esg': 'CC',
        'MINERvA:2022djk' : 'CC',
        'MINERvA:2014ogb': 'CC',
        'MINERvA:2016sfc': 'CC',
        'MINERvA:2014ani': 'CC',
        'MINERvA:2019rhx': 'CC',
        'MINERvA:2017ipy': 'CC',
        'MiniBooNE:2010eis': 'CC',
        'MiniBooNE:2009koj': 'CC',
        'SciBooNE:2008bzb': 'CC',
        'T2K:2023xlh': 'CC Coh νμ+ν̄μ',
        'T2K:2021naz': 'CC TKI',
        'T2K:2016cbz': 'CC',
        'T2K:2016soz': 'CC',
        'T2K:2019yqu': 'CC',
        'NINJA:2022zbi': 'CC event rates on Fe',
        'NOMAD:2001vdk': 'CC bwd π- and p'
    }
    
    neutral_pion_refs = {
        'MicroBooNE:2024sec': 'NC',
        'MicroBooNE:2024bnl': 'CC',
        'ArgoNeuT:2015ldo': 'NC',
        'K2K:2010xeb': 'CC',
        'K2K:2004qpv': 'NC',
        'MicroBooNE:2018neo': 'CC',
        'MicroBooNE:2022zhr': 'NC',
        'MINERvA:2020anu': 'CC',
        'MINERvA:2016sfc': 'CC',
        'MINERvA:2015slz': 'CC',
        'MINERvA:2017okh': 'CC',
        'MINERvA:2016uck': 'NC',
        'MiniBooNE:2010cxl': 'CC',
        'MiniBooNE:2009dxl': 'NC',
        'MiniBooNE:2008mmr': 'NC',
        'MINOS:2016yyz': 'NC',
        'NOMAD:2009idt': 'NC',
        'NOvA:2019bdw': 'NC',
        'NOvA:2023uxq': 'NC',
        'SciBooNE:2009nlf': 'NC',
        'SciBooNE:2010lca': 'NC',
        'T2K:2017epu': 'NC'
    }

    exotic = {
        'MINERvA:2016ymg': 'NC K+',
        'MINERvA:2016zyp': 'CC K+',
        'MINERvA:2016cun': 'CC-Coh K+',
        'SciBooNE:2011sjq': 'CC K+',
        'T2K:2019odo': 'CC gamma',
        'MicroBooNE:2023ubu': 'CC+NC eta',
        'MicroBooNE:2022cls': 'CC Lambda',
        'NOMAD:2011gyy': 'CC gamma',
        'NOMAD:2004djf': 'NC K0, K+-, Λ, Λ ̄, Σ+-',
        'NOMAD:2001kdr': 'CC K0, K+-, Λ, Λ ̄, Σ+-, Σ0, Ξ-',
        'NOMAD:2001kpt': 'CC D+',
        'NOMAD:2001ozt': 'CC ρ0, f0, f2',
        'NOMAD:2001iup': 'CC Λ ̄',
        'NOMAD:2000wdf': 'CC Λ',
        'NOMAD:1998gsx': 'CC DIS',
    }

In [17]:
len(charged_pion_refs), len(neutral_pion_refs)

(21, 22)